In [8]:
!pip install scikit-learn
!pip install pandas

/Users/balazs/anaconda3/envs/pcx/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 6.2 MB/s eta 0:00:0000:0100:01


In [11]:
# MNIST Classification with Predictive Coding Networks

import jax
import jax.numpy as jnp
import equinox as eqx
import pcx as px
import pcx.predictive_coding as pxc
import pcx.nn as pxnn
import pcx.functional as pxf
import pcx.utils as pxu

import optax
from typing import Callable
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

# Load MNIST using scikit-learn
def get_datasets(batch_size: int):
    X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
    X = X.astype('float32') / 255.0
    y = y.astype('int32')
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Create batch indices
    def batch_indices(length, batch_size):
        indices = np.arange(length)
        np.random.shuffle(indices)
        return np.array_split(indices, length // batch_size)
    
    train_batches = [(X_train[idx], y_train[idx]) for idx in batch_indices(len(X_train), batch_size)]
    test_batches = [(X_test[idx], y_test[idx]) for idx in batch_indices(len(X_test), batch_size)]
    
    return train_batches, test_batches

# Rest of the code remains the same...

# Define the model
class MNISTModel(pxc.EnergyModule):
    def __init__(
        self,
        hidden_dims: list[int],
        act_fn: Callable[[jax.Array], jax.Array] = jax.nn.relu,
        key: px.random.PRNGKey = px.random.PRNGKey(0)  # Use px.random instead of jax.random
    ) -> None:
        super().__init__()
        
        self.act_fn = px.static(act_fn)
        
        # Create layers
        input_dim = 28 * 28  # Flattened MNIST image
        layer_dims = [input_dim] + hidden_dims + [10]  # 10 classes for MNIST
        
        # Split key for each layer
        keys = px.random.split(key, len(layer_dims) - 1)
        
        self.layers = []
        for key, in_dim, out_dim in zip(keys, layer_dims[:-1], layer_dims[1:]):
            self.layers.append(pxnn.Linear(in_dim, out_dim, key=key))
            
        # Create VODEs
        self.vodes = [pxc.Vode() for _ in range(len(self.layers)-1)]
        self.vodes.append(pxc.Vode(pxc.ce_energy))  # Cross entropy for final layer
        self.vodes[-1].h.frozen = True

    def __call__(self, x, y):
        # Ensure input is properly shaped [batch_size, features]
        batch_size = x.shape[0]
        x = x.reshape(batch_size, -1)  # Flatten while preserving batch dimension
        
        # Forward pass through hidden layers
        for v, l in zip(self.vodes[:-1], self.layers[:-1]):
            x = v(self.act_fn(l(x)))
            
        # Output layer
        x = self.vodes[-1](self.layers[-1](x))
        
        if y is not None:
            self.vodes[-1].set("h", y)
            
        return self.vodes[-1].get("u")
        

# Define forward and energy functions
@pxf.vmap(pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)), in_axes=(0, 0), out_axes=0)
def forward(x, y, *, model: MNISTModel):
    return model(x, y)

@pxf.vmap(pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)), in_axes=(0,), out_axes=(None, 0), axis_name="batch")
def energy(x, *, model: MNISTModel):
    y_ = model(x, None)
    return jax.lax.psum(model.energy(), "batch"), y_

# Training function for one batch
@pxf.jit(static_argnums=0)
def train_on_batch(
    T: int,
    x: jax.Array,
    y: jax.Array,
    *,
    model: MNISTModel,
    optim_w: pxu.Optim,
    optim_h: pxu.Optim
):
    model.train()
    
    # Initialize
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        forward(x, y, model=model)
    
    # Inference steps
    for _ in range(T):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (e, y_), g = pxf.value_and_grad(
                pxu.M_hasnot(pxc.VodeParam, frozen=True).to([False, True]),
                has_aux=True
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
    
    # Weight update
    with pxu.step(model, clear_params=pxc.VodeParam.Cache):
        (e, y_), g = pxf.value_and_grad(
            pxu.M(pxnn.LayerParam).to([False, True]), 
            has_aux=True
        )(energy)(x, model=model)
    optim_w.step(model, g["model"], scale_by=1.0/x.shape[0])

# Evaluation function
@pxf.jit()
def eval_on_batch(x: jax.Array, y: jax.Array, *, model: MNISTModel):
    model.eval()
    
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        y_ = forward(x, None, model=model).argmax(axis=-1)
    
    return (y_ == y).mean()


# Main training loop
def main():
    # Hyperparameters
    batch_size = 128
    hidden_dims = [512, 256]  # Two hidden layers
    h_lr = 0.01
    w_lr = 0.001
    num_epochs = 10
    T = 8  # Number of inference steps
    
    # Initialize model with random key
    key = px.random.PRNGKey(0)  # Use px.random instead of jax.random
    model = MNISTModel(hidden_dims, key=key)
    
    # Initialize optimizers
    optim_h = pxu.Optim(
        lambda: optax.sgd(h_lr), 
        pxu.M_hasnot(pxc.VodeParam, frozen=True)(model)
    )
    optim_w = pxu.Optim(
        lambda: optax.adam(w_lr), 
        pxu.M(pxnn.LayerParam)(model)
    )
    
    # Get data
    train_batches, test_batches = get_datasets(batch_size)
    
    # Training loop
    for epoch in range(num_epochs):
        # Training
        for x, y in train_batches:
            x = jnp.array(x)
            y = jax.nn.one_hot(jnp.array(y, dtype=jnp.int32), 10)
            train_on_batch(T, x, y, model=model, optim_w=optim_w, optim_h=optim_h)
        
        # Evaluation
        test_acc = []
        for x, y in test_batches:
            x = jnp.array(x)
            y = jnp.array(y, dtype=jnp.int32)
            acc = eval_on_batch(x, y, model=model)
            test_acc.append(acc)
        
        print(f"Epoch {epoch+1}, Test accuracy: {np.mean(test_acc):.4f}")

if __name__ == "__main__":
    main()

AttributeError: module 'pcx' has no attribute 'random'